<a href="https://colab.research.google.com/github/ranimiftah12345678/ML_22_Ranimiftah/blob/main/JS03/tugasJS03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

In [2]:
# 1. Load data & pisahkan variabel
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y_raw = np.array(['M' if t == 0 else 'B' for t in data.target])

In [3]:
# 2. Encoding kolom diagnosis
le = LabelEncoder()
y = le.fit_transform(y_raw)  # B=0, M=1

X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

In [4]:
# 3-6. Standardisasi + Seleksi Fitur + Logistic Regression dalam Pipeline
# Uji berbagai nilai k untuk cari jumlah fitur terbaik
for k in [5, 10, 15, 20, 25, 30]:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("sel", SelectKBest(score_func=f_classif, k=k)),
        ("clf", LogisticRegression(max_iter=5000))
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    print(f"k={k:2d}  accuracy={accuracy_score(y_test, pred):.4f}")

k= 5  accuracy=0.9649
k=10  accuracy=0.9561
k=15  accuracy=0.9737
k=20  accuracy=0.9825
k=25  accuracy=0.9737
k=30  accuracy=0.9649


In [5]:
best_k = int(pd.DataFrame([
    {"k": 5, "accuracy": 0.9649},
    {"k": 10, "accuracy": 0.9561},
    {"k": 15, "accuracy": 0.9737},
    {"k": 20, "accuracy": 0.9825},
    {"k": 25, "accuracy": 0.9737},
    {"k": 30, "accuracy": 0.9649},
]).loc[lambda d: d["accuracy"].idxmax(), "k"])

print("k terbaik:", best_k)

pipe_best = Pipeline([
    ("scaler", StandardScaler()),
    ("sel", SelectKBest(score_func=f_classif, k=best_k)),
    ("clf", LogisticRegression(max_iter=5000))
])
pipe_best.fit(X_train, y_train)
pred_best = pipe_best.predict(X_test)
print(classification_report(y_test, pred_best, target_names=le.classes_))

mask = pipe_best.named_steps["sel"].get_support()
selected_features = X.columns[mask]
scores = pipe_best.named_steps["sel"].scores_[mask]
df_fitur = pd.DataFrame({"fitur": selected_features, "skor": scores}).sort_values("skor", ascending=False)
print(df_fitur)

k terbaik: 20
              precision    recall  f1-score   support

           B       0.97      1.00      0.99        72
           M       1.00      0.95      0.98        42

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

                   fitur        skor
18  worst concave points  733.724933
13       worst perimeter  717.246487
11          worst radius  692.861395
6    mean concave points  684.526845
2         mean perimeter  548.413236
14            worst area  522.188947
0            mean radius  511.274848
3              mean area  444.857518
5         mean concavity  397.592082
17       worst concavity  319.507776
4       mean compactness  263.564326
16     worst compactness  238.203725
7           radius error  205.433206
8        perimeter error  193.168931
9             area error  180.561229
12         worst texture  126.119113
15      worst smoothness  103.